# T3: タイヤデグラデーション分析
## F1 2026 R01–R03 クリーンロングランデータを使ったタイヤ劣化解析

**入力**: `notebooks/output/clean_longruns.csv`（T2で抽出したクリーンロングラン）  
**比較**: `data/cross_gp_analysis/csv/cross_gp_deg_rates.csv`（107%フィルタ前）  
**出力**:
- `notebooks/output/deg_rates_clean.csv` — ロングラン別デグレートテーブル
- `notebooks/output/deg_heatmap.png` — チーム×コンパウンドヒートマップ

---

### 分析内容
1. 各ロングランに対して `TyreLife vs LapTime_sec` の線形回帰
2. 傾き（slope） = **デグレート（秒/ラップ）**
   - 正の値 → タイヤ劣化が優勢
   - 負の値 → 燃料軽量化効果が優勢
3. コンパウンド別×チーム別×GP別のデグレートテーブル
4. 107%フィルタ前後の比較
5. 燃料効果の考察
6. チーム別タイヤマネジメント評価

In [ ]:
## セル 1: インポートと設定
import matplotlib
matplotlib.use('Agg')  # GUIなし環境向け（Jupyter実行時はコメントアウト可）

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import linregress
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')

# ── パス設定 ──
NOTEBOOK_DIR  = os.getcwd()  # notebooks/
PROJECT_DIR   = os.path.dirname(NOTEBOOK_DIR)
INPUT_CSV     = os.path.join(NOTEBOOK_DIR, 'output', 'clean_longruns.csv')
OLD_CSV       = os.path.join(PROJECT_DIR, 'data', 'cross_gp_analysis', 'csv', 'cross_gp_deg_rates.csv')
OUTPUT_DIR    = os.path.join(NOTEBOOK_DIR, 'output')
OUTPUT_CSV    = os.path.join(OUTPUT_DIR, 'deg_rates_clean.csv')
OUTPUT_HEATMAP = os.path.join(OUTPUT_DIR, 'deg_heatmap.png')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── グラフスタイル（CLAUDE.md準拠） ──
STYLE = {
    'bg_color':   '#1a1a2e',
    'text_color': '#ffffff',
    'grid_color': '#333355',
    'figsize':    (14, 8),
    'title_size': 16,
    'label_size': 11,
}

# 日本語フォント（Hiragino Sans: macOS標準）
available = {f.name for f in fm.fontManager.ttflist}
JP_FONT = 'Hiragino Sans' if 'Hiragino Sans' in available else None
if JP_FONT:
    plt.rcParams['font.family'] = JP_FONT

# タイヤコンパウンドカラー
COMPOUND_COLORS = {
    'SOFT':   '#FF3333',
    'MEDIUM': '#FFD700',
    'HARD':   '#FFFFFF',
}

# デグレート区分（秒/ラップ）
DEG_LOW    = 0.05   # 低デグ境界
DEG_MEDIUM = 0.10   # 中デグ境界
MIN_LAPS   = 5      # ロングラン最小周回数

def classify_deg(rate: float) -> str:
    """デグレートを低/中/高に分類する"""
    abs_rate = abs(rate)
    if abs_rate < DEG_LOW:
        return '低デグ'
    elif abs_rate < DEG_MEDIUM:
        return '中デグ'
    return '高デグ'

print("セットアップ完了")
print(f"  入力: {INPUT_CSV}")
print(f"  出力先: {OUTPUT_DIR}")

In [ ]:
## セル 2: データ読み込み
df = pd.read_csv(INPUT_CSV)
print(f"クリーンロングランデータ: {len(df)}行")
print(f"ロングランID数: {df['LongRunID'].nunique()}")
print(f"GP: {list(df['GP'].unique())}")
print(f"コンパウンド: {list(df['Compound'].unique())}")
print(f"\n先頭5行:")
df.head()

In [ ]:
## セル 3: ロングラン別線形回帰（TyreLife vs LapTime_sec）
# 各ロングランIDに対して線形回帰を実行
# slope（傾き）= デグレート（秒/ラップ）
# R² = 回帰の決定係数（信頼性の目安）

results = []
skipped = 0

for run_id, group in df.groupby('LongRunID'):
    # 5周以上のみ回帰対象
    if len(group) < MIN_LAPS:
        skipped += 1
        continue

    x = group['TyreLife'].values.astype(float)
    y = group['LapTime_sec'].values.astype(float)

    # NaN除外
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    if len(x) < MIN_LAPS:
        skipped += 1
        continue

    # scipy 線形回帰
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    r2 = r_value ** 2

    meta = group.iloc[0]
    results.append({
        'GP':          meta['GP'],
        'Driver':      meta['Driver'],
        'Team':        meta['Team'],
        'Stint':       meta['Stint'],
        'Compound':    meta['Compound'],
        'DegRate':     round(slope, 6),      # 秒/ラップ
        'Intercept':   round(intercept, 3),
        'R2':          round(r2, 4),
        'PValue':      round(p_value, 4),
        'CleanLaps':   len(x),
        'MeanPace':    round(np.mean(y), 4),
        'MinTyreLife': int(x.min()),
        'MaxTyreLife': int(x.max()),
        'LongRunID':   run_id,
    })

df_result = pd.DataFrame(results)
df_result['DegClass'] = df_result['DegRate'].apply(classify_deg)
df_result['GPShort']  = df_result['GP'].str.extract(r'(R\d+)')[0]

print(f"回帰完了: {len(df_result)}ロングラン（スキップ: {skipped}件）")
print(f"平均R²: {df_result['R2'].mean():.3f}")
print(f"\nデグレート分布:")
print(df_result[['GP', 'Driver', 'Team', 'Compound', 'DegRate', 'R2', 'CleanLaps']].head(10).to_string(index=False))